# Margin Trading

Until now, our backtests have been limited to cash-only trades. This notebook explores the margin trading features in PyBroker v2. We'll use [StrategyConfig](https://www.pybroker.com/en/latest/reference/pybroker.config.html#pybroker.config.StrategyConfig) to apply leverage to our buying power, calculate interest on borrowed funds, and set up collateral for short positions.

In [1]:
import pybroker
from pybroker import Strategy, StrategyConfig, YFinance, sumv

pybroker.enable_data_source_cache("margin_trading")

## Configuring Leverage

Margin is enabled with the `leverage` config option, which multiplies buying power for both long and short positions. The default of `1.0` buys with cash only. Setting it to `2.0` allows holding positions worth up to 2x our equity, and the borrowed remainder is tracked as a margin loan. PyBroker does not model margin calls. Orders are instead limited to the available buying power at fill time:

In [2]:
config = StrategyConfig(initial_cash=100_000, leverage=2.0)

Next, we define a simple trend-following rule: hold a symbol while it closes above its 50-day moving average.

Position sizing is where leverage takes effect. [set_target_shares](https://www.pybroker.com/en/latest/reference/pybroker.context.html#pybroker.context.ExecContext.set_target_shares) sizes orders as a fraction of deployable capital. This capital equals our equity multiplied by `leverage`. As a result, targeting 25% across four stocks deploys up to roughly 2x our equity when trading on margin:

In [3]:
sma_50 = pybroker.indicator("sma_50", lambda data: sumv(data.close, 50) / 50)


def trend_follow(ctx):
    sma = ctx.indicator("sma_50")[-1]
    if ctx.long_pos() is None and ctx.close[-1] > sma:
        ctx.set_target_shares(0.25, dir="long")
    elif ctx.long_pos() is not None and ctx.close[-1] < sma:
        ctx.sell_all_shares()


yfinance = YFinance()
strategy = Strategy(
    YFinance(), start_date="1/1/2021", end_date="1/1/2026", config=config
)
strategy.add_execution(
    trend_follow, ["GS", "MS", "C", "USB"], indicators=sma_50
)
result_2x = strategy.backtest()

Backtesting: 2021-01-01 00:00:00 to 2026-01-01 00:00:00

Loading bar data...


[*********************100%***********************]  4 of 4 completed

Loaded bar data: 0:00:01 



Computing indicators...


100% (4 of 4) |##########################| Elapsed Time: 0:00:00 Time:  0:00:00



Test split: 2021-01-04 00:00:00 to 2025-12-31 00:00:00


100% (1255 of 1255) |####################| Elapsed Time: 0:00:00 Time:  0:00:000:00



Finished backtest: 0:00:01


`result.portfolio` records the margin balances on every bar. When a levered position is opened, a portion of cash (`notional / leverage`) is posted as collateral. The borrowed remainder is tracked in `margin_loan`, and the `net_cash_balance` equals `cash - margin_loan`.

To see the margin mechanics in effect, we filter the output to show only bars with an outstanding loan.

On the first of these filtered bars, a single entry was filled. Cash dropped by `$25,413` to post half of the roughly `$51,000` notional as collateral. The `margin_loan` column carries the borrowed half. Finally, the `notional` column tracks the position's total exposure as it is marked to each close:

In [4]:
levered = result_2x.portfolio[result_2x.portfolio["margin_loan"] > 0]
levered[
    [
        "cash",
        "equity",
        "notional",
        "margin_loan",
        "net_cash_balance",
        "market_value",
    ]
].head()

,cash,equity,notional,margin_loan,net_cash_balance,market_value
date,,,,,,
2021-03-17,12.58,100830.65,200805.50,99987.43,-99974.85,100830.65
2021-03-18,12.58,103107.59,203082.44,99987.43,-99974.85,103107.59
2021-03-19,12.58,100417.96,200392.81,99987.43,-99974.85,100417.96
2021-03-22,12.58,97492.09,197466.94,99987.43,-99974.85,97492.09
2021-03-23,12.58,93378.68,193353.53,99987.43,-99974.85,93378.68


## Comparing Against Cash-Only

To isolate the effects of leverage, we rerun the strategy with a cash-only `leverage` of `1.0`. We also disable PyBroker's logging to keep the output clean for the remaining runs:

In [5]:
pybroker.disable_logging()


def run_backtest(
    config, exec_fn=trend_follow, symbols=("GS", "MS", "C", "USB")
):
    strategy = Strategy(
        yfinance, start_date="1/1/2021", end_date="1/1/2026", config=config
    )
    strategy.add_execution(exec_fn, symbols, indicators=sma_50)
    return strategy.backtest()


result_1x = run_backtest(StrategyConfig(initial_cash=100_000))
print(f"1x total return: {result_1x.metrics.total_return_pct:.2f}%")
print(f"2x total return: {result_2x.metrics.total_return_pct:.2f}%")
print(f"1x max drawdown: {result_1x.metrics.max_drawdown_pct:.2f}%")
print(f"2x max drawdown: {result_2x.metrics.max_drawdown_pct:.2f}%")

1x total return: 6.53%
2x total return: -0.34%
1x max drawdown: -40.59%
2x max drawdown: -65.64%


## Charging Margin Interest

Leverage more than doubled the return and nearly doubled the max drawdown, but borrowing is not free. The `interest_rate` config option models this financing cost. It applies an annual percentage rate to the portfolio's net cash balance, accruing once per bar at `interest_rate / bars_per_year`. Note that using this feature requires `bars_per_year` to be set.

Interest is charged when net cash is negative (the margin loan exceeds cash) and credited when net cash is positive. To isolate this financing cost, we will backtest a buy-and-hold execution. Because both runs will hold identical positions, the entire difference in their final market value represents the interest paid:

In [6]:
def buy_and_hold(ctx):
    if ctx.long_pos() is None:
        ctx.set_target_shares(0.25, dir="long")


config_interest = StrategyConfig(
    initial_cash=100_000,
    leverage=2.0,
    interest_rate=6.0,
    bars_per_year=252,
)
result_hold = run_backtest(config, buy_and_hold)
result_interest = run_backtest(config_interest, buy_and_hold)
print(
    "Final market value (no interest):",
    result_hold.portfolio["market_value"].iloc[-1],
)
print(
    "Final market value (6% interest):",
    result_interest.portfolio["market_value"].iloc[-1],
)

Final market value (no interest): 348041.01
Final market value (6% interest): 313281.93


Looking at the tail of the portfolio, cash remains pinned at zero. Meanwhile, accrued interest is added to the `margin_loan`, causing it to increase with every bar:

In [7]:
result_interest.portfolio[
    ["cash", "margin_loan", "net_cash_balance", "market_value"]
].tail()

,cash,margin_loan,net_cash_balance,market_value
date,,,,
2025-12-24,0.0,134666.95,-134666.95,328104.14
2025-12-26,0.0,134699.02,-134699.02,326634.98
2025-12-29,0.0,134731.09,-134731.09,319980.42
2025-12-30,0.0,134763.17,-134763.17,316896.59
2025-12-31,0.0,134795.25,-134795.25,313281.93


## Shorting on Margin

Short selling also uses margin. Opening a short position posts collateral equal to `notional / leverage`, and shorts draw from the same shared buying power as long positions. 

The portfolio's `margin` column tracks the notional exposure of open short positions at the current market price. Inside an execution, these same balances are accessible using [ctx.buying_power](https://www.pybroker.com/en/latest/reference/pybroker.context.html#pybroker.context.ExecContext.buying_power), `ctx.margin_loan`, and `ctx.total_margin`. 

To demonstrate this, we invert our trend rule to short symbols trading below their moving average. We will use symbols that spent more of the backtest below trend, and we will enable `record_position_bars` to capture per-position balances:

In [8]:
def trend_short(ctx):
    sma = ctx.indicator("sma_50")[-1]
    if ctx.short_pos() is None and ctx.close[-1] < sma:
        ctx.set_target_shares(0.25, dir="short")
    elif ctx.short_pos() is not None and ctx.close[-1] > sma:
        ctx.cover_all_shares()


config_short = StrategyConfig(
    initial_cash=100_000, leverage=2.0, record_position_bars=True
)
result_short = run_backtest(
    config_short, trend_short, ["HD", "LOW", "CMCSA", "KHC"]
)

On bars with open short positions, the `margin` column carries their notional exposure. Under leverage, this exposure can exceed your total equity. Collateral equal to `notional / leverage` is posted from cash, while the `margin_loan` carries the borrowed remainder of the shorted notional. The `net_cash_balance` subtracts this loan from your remaining cash, dropping into the negative once the loan exceeds available cash. 

Finally, note that `equity` holds shorts at their entry cost, whereas `market_value` includes their unrealized PnL:

In [9]:
shorted = result_short.portfolio[result_short.portfolio["margin"] > 0]
shorted[
    [
        "cash",
        "equity",
        "margin",
        "margin_loan",
        "net_cash_balance",
        "market_value",
    ]
].head()

,cash,equity,margin,margin_loan,net_cash_balance,market_value
date,,,,,,
2021-04-12,75095.15,100000.0,49722.03,24904.85,50190.3,100087.67
2021-04-13,75095.15,100000.0,50244.24,24904.85,50190.3,99565.46
2021-04-14,75095.15,100000.0,49703.37,24904.85,50190.3,100106.33
2021-04-15,75095.15,100000.0,50486.69,24904.85,50190.3,99323.01
2021-04-16,75095.15,100000.0,51391.24,24904.85,50190.3,98418.46


Because we enabled `record_position_bars`, `result.positions` tracks the balances for each individual position. This includes each short's specific share of the portfolio's `margin`, as well as its own unrealized PnL:

In [10]:
result_short.positions[
    ["short_shares", "close", "margin", "unrealized_pnl"]
].head()

short_shares  close    margin  unrealized_pnl
symbol date                                                     
CMCSA  2021-04-12           995  49.97  49722.03           87.67
       2021-04-13           995  50.50  50244.24         -434.54
       2021-04-14           995  49.95  49703.37          106.33
       2021-04-15           995  50.74  50486.69         -676.99
       2021-04-16           995  51.65  51391.24        -1581.54